# 07 — Xception41p 5-Fold OOF + DenseNet Ensemble

Mục tiêu:
- Xác nhận Xception41p pretrained @ 299 trên toàn bộ development set bằng 5-fold OOF.
- Giữ internal holdout 770 ảnh **niêm phong tuyệt đối**.
- Preload pad+resize toàn bộ dev set một lần vào RAM để tránh decode/resize JPEG lặp lại mỗi fold.
- Tạo OOF predictions, fine-scan threshold 0.001.
- Ghép OOF Xception với DenseNet121 OOF, tối ưu weight + threshold trên OOF và kiểm tra độ ổn định theo từng fold.

Cấu hình khởi đầu kế thừa từ fold-0 screen:
- `xception41p.ra3_in1k` (ImageNet pretrained via timm)
- 299×299
- physical batch 4 + gradient accumulation 2 (effective batch 8)
- weighted cross-entropy
- AdamW, cosine LR
- mild affine + brightness/contrast augmentation
- max 10 epochs, patience 3

> Không dùng holdout cho architecture / threshold / ensemble tuning.


In [ ]:
print('>>> CELL 1 START: imports/config', flush=True)
import os
os.environ['TORCH_HOME']=r'D:\AI_Cache\torch'
os.environ['HF_HOME']=r'D:\AI_Cache\huggingface'

from pathlib import Path
import json, time, random, gc
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
import timm

SEED=2026
ROOT=Path(r'D:\medicine')
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
OUT=ROOT/'cv_outputs'/'xception41p_299'
DENSE_OOF=ROOT/'cv_outputs'/'densenet121_224'/'oof_predictions.csv'
OUT.mkdir(parents=True,exist_ok=True)

IMAGE_SIZE=299
BATCH_SIZE=4
GRAD_ACCUM_STEPS=2
EPOCHS=10
PATIENCE=3
LR=1e-4
WD=1e-4
PROGRESS_EVERY=50
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
torch.backends.cudnn.benchmark=USE_AMP

print('PyTorch:',torch.__version__, flush=True)
print('Device:',DEVICE, flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0), flush=True)
print('timm:',timm.__version__, flush=True)
print('Batch:',BATCH_SIZE,'| accum:',GRAD_ACCUM_STEPS,'| effective:',BATCH_SIZE*GRAD_ACCUM_STEPS, flush=True)
print('Max epochs:',EPOCHS,'| patience:',PATIENCE, flush=True)


In [ ]:
print('>>> CELL 2 START: load locked split', flush=True)
m=pd.read_csv(MANIFEST)
dev=m[m.split=='dev'].copy().reset_index(drop=True)
holdout=m[m.split=='holdout'].copy().reset_index(drop=True)
assert len(dev)==4362 and len(holdout)==770
assert set(dev.sha256).isdisjoint(set(holdout.sha256))
assert dev.cv_fold.notna().all()
dev['cache_idx']=np.arange(len(dev),dtype=np.int32)

print('Dev:',len(dev),dev.label.value_counts().to_dict(), flush=True)
print('Fold sizes:',dev.cv_fold.value_counts().sort_index().to_dict(), flush=True)
print('SEALED holdout:',len(holdout),'(NOT USED)', flush=True)


In [ ]:
print('>>> CELL 3 START: preload dev images once into RAM', flush=True)

class PadToSquare:
    def __call__(self,img):
        w,h=img.size; s=max(w,h)
        l=(s-w)//2; r=s-w-l; t=(s-h)//2; b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

pad=PadToSquare()
t0=time.time()
cache=[]
print(f'Preloading {len(dev)} dev images -> {IMAGE_SIZE}x{IMAGE_SIZE} uint8...', flush=True)
for i,row in dev.iterrows():
    with Image.open(row.path) as im:
        im=im.convert('RGB')
        im=pad(im)
        im=im.resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BILINEAR)
        cache.append(np.asarray(im,dtype=np.uint8))
    if (i+1)%500==0 or (i+1)==len(dev):
        print(f'  cached {i+1}/{len(dev)}', flush=True)

DEV_CACHE=np.stack(cache,axis=0)
del cache
print(f'Cache shape={DEV_CACHE.shape} | RAM={DEV_CACHE.nbytes/1024**3:.2f} GiB | {time.time()-t0:.0f}s', flush=True)

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
train_tf=transforms.Compose([
    transforms.RandomAffine(degrees=5,translate=(0.02,0.02),scale=(0.95,1.05),fill=0),
    transforms.ColorJitter(brightness=0.08,contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])
val_tf=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])
L2I={'Normal':0,'Pneumonia':1}

class CachedXrayDataset(Dataset):
    def __init__(self,frame,tf):
        self.df=frame.reset_index(drop=True)
        self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        im=Image.fromarray(DEV_CACHE[int(row.cache_idx)])
        return self.tf(im),L2I[row.label],row.filename


In [ ]:
print('>>> CELL 4 START: metrics/helpers', flush=True)

def metrics(y_true,probs,threshold=0.5):
    y_true=np.asarray(y_true,dtype=int)
    probs=np.asarray(probs,dtype=float)
    pred=(probs>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y_true)
    se=tp/max(tp+fn,1)
    sp=tn/max(tn+fp,1)
    f1=f1_score(y_true,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    auc=roc_auc_score(y_true,probs)
    return {
        'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),
        'Specificity':float(sp),'F1':float(f1),'AUROC':float(auc),
        'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
        'threshold':float(threshold)
    }

@torch.no_grad()
def predict(model,loader):
    model.eval(); ys=[]; ps=[]; names=[]
    for x,y,n in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x)
        p=torch.softmax(logits.float(),dim=1)[:,1]
        ys.extend(y.numpy())
        ps.extend(p.cpu().numpy())
        names.extend(list(n))
    return np.asarray(ys),np.asarray(ps),names

def make_model():
    print('  creating Xception41p pretrained model...', flush=True)
    return timm.create_model('xception41p.ra3_in1k',pretrained=True,num_classes=2).to(DEVICE)

def threshold_scan(y,p,step=0.001):
    ts=np.round(np.arange(0.05,0.9501,step),3)
    df=pd.DataFrame([metrics(y,p,t) for t in ts])
    best=df.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
    return df,best


## Train 5 folds

Mỗi fold dùng seed `2026 + fold`. Exact duplicate SHA-256 groups đã được khóa trong cùng fold từ split protocol.


In [ ]:
print('>>> CELL 6 START: Xception 5-fold training', flush=True)
all_oof=[]
fold_summaries=[]

for fold in range(5):
    print('\n'+'='*78, flush=True)
    print(f'FOLD {fold}/4', flush=True)
    print('='*78, flush=True)

    seed=SEED+fold
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if USE_AMP: torch.cuda.manual_seed_all(seed)

    train_df=dev[dev.cv_fold!=fold].copy().reset_index(drop=True)
    val_df=dev[dev.cv_fold==fold].copy().reset_index(drop=True)
    assert set(train_df.sha256).isdisjoint(set(val_df.sha256))
    print('Train:',len(train_df),'Val:',len(val_df), flush=True)

    train_loader=DataLoader(
        CachedXrayDataset(train_df,train_tf),
        batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=False
    )
    val_loader=DataLoader(
        CachedXrayDataset(val_df,val_tf),
        batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=False
    )

    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()
    model=make_model()

    c=train_df.label.value_counts()
    n0=int(c.Normal); n1=int(c.Pneumonia); total=n0+n1
    class_w=torch.tensor([total/(2*n0),total/(2*n1)],dtype=torch.float32,device=DEVICE)
    criterion=nn.CrossEntropyLoss(weight=class_w)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)

    best_score=-1.0; best_epoch=0; best_state=None; stale=0; hist=[]

    for epoch in range(1,EPOCHS+1):
        et0=time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss_sum=0.0; seen=0

        for batch_idx,(x,y,_) in enumerate(train_loader,1):
            x=x.to(DEVICE,non_blocking=True)
            y=y.to(DEVICE,non_blocking=True)

            with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
                logits=model(x)
                raw_loss=criterion(logits,y)
                loss=raw_loss/GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            is_step=(batch_idx % GRAD_ACCUM_STEPS==0) or (batch_idx==len(train_loader))
            if is_step:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            loss_sum+=float(raw_loss.item())*len(x)
            seen+=len(x)

            if batch_idx%PROGRESS_EVERY==0 or batch_idx==len(train_loader):
                print(
                    f'  fold {fold} epoch {epoch:02d} batch {batch_idx}/{len(train_loader)} '
                    f'| avg_loss={loss_sum/seen:.4f}',
                    flush=True
                )

        scheduler.step()
        yv,pv,_=predict(model,val_loader)
        mm=metrics(yv,pv,0.5)
        train_loss=loss_sum/seen
        sec=time.time()-et0
        hist.append({'fold':fold,'epoch':epoch,'train_loss':train_loss,'seconds':sec,**mm})

        print(
            f"Fold {fold} Epoch {epoch:02d} | loss={train_loss:.4f} | "
            f"P={mm['P']:.5f} | A={mm['Accuracy']:.5f} | "
            f"Se={mm['Sensitivity']:.5f} | Sp={mm['Specificity']:.5f} | "
            f"F1={mm['F1']:.5f} | AUROC={mm['AUROC']:.6f} | {sec:.0f}s",
            flush=True
        )

        if mm['P']>best_score+1e-6:
            best_score=mm['P']; best_epoch=epoch; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if stale>=PATIENCE:
                print(f'Fold {fold}: early stopping after epoch {epoch}', flush=True)
                break

    model.load_state_dict(best_state)
    yv,pv,names=predict(model,val_loader)
    fold_m=metrics(yv,pv,0.5)

    pred_df=pd.DataFrame({
        'filename':names,'y_true':yv,'p_pneumonia':pv,'fold':fold
    })
    pred_df.to_csv(OUT/f'fold{fold}_predictions.csv',index=False)
    pd.DataFrame(hist).to_csv(OUT/f'fold{fold}_history.csv',index=False)
    torch.save({
        'architecture':'xception41p.ra3_in1k',
        'pretrained':'ImageNet',
        'fold':fold,
        'image_size':IMAGE_SIZE,
        'best_epoch':best_epoch,
        'state_dict':best_state
    },OUT/f'fold{fold}_best.pt')

    fold_summaries.append({'fold':fold,'best_epoch':best_epoch,**fold_m})
    all_oof.append(pred_df)
    print('BEST fold metrics:',fold_m, flush=True)

    del model,optimizer,scheduler,scaler,best_state,train_loader,val_loader
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

print('>>> ALL 5 FOLDS FINISHED', flush=True)


## Xception OOF score + fine threshold scan


In [ ]:
print('>>> CELL 8 START: Xception OOF evaluation', flush=True)
oof=pd.concat(all_oof,ignore_index=True)
assert len(oof)==len(dev)==4362
assert oof.filename.nunique()==len(oof)
oof.to_csv(OUT/'oof_predictions.csv',index=False)

fold_table=pd.DataFrame(fold_summaries)
fold_table.to_csv(OUT/'fold_summary.csv',index=False)
display(fold_table)

x_base=metrics(oof.y_true,oof.p_pneumonia,0.5)
x_scan,x_best=threshold_scan(oof.y_true,oof.p_pneumonia,0.001)
x_best_t=float(x_best.threshold)
x_tuned=metrics(oof.y_true,oof.p_pneumonia,x_best_t)
x_scan.to_csv(OUT/'oof_threshold_scan_001.csv',index=False)

print('Fold P mean:',fold_table.P.mean(),'std:',fold_table.P.std(ddof=1), flush=True)
print('Xception OOF @0.5:',x_base, flush=True)
print('Xception OOF fine-tuned:',x_tuned, flush=True)


## DenseNet + Xception OOF ensemble

Tối ưu weight và threshold **chỉ trên OOF**. Làm coarse search trước, sau đó refine quanh vùng tốt nhất.


In [ ]:
print('>>> CELL 10 START: DenseNet + Xception OOF ensemble', flush=True)
assert DENSE_OOF.exists(),f'Missing DenseNet OOF: {DENSE_OOF}'
dense=pd.read_csv(DENSE_OOF)[['filename','y_true','p_pneumonia','fold']].rename(
    columns={'p_pneumonia':'p_dense','fold':'dense_fold'}
)
x=oof.rename(columns={'p_pneumonia':'p_xception','fold':'x_fold'})
z=dense.merge(x[['filename','y_true','p_xception','x_fold']],on=['filename','y_true'])
assert len(z)==4362
assert (z.dense_fold.astype(int)==z.x_fold.astype(int)).all()

# Fine standalone DenseNet baseline using the same 0.001 threshold resolution.
d_scan,d_best=threshold_scan(z.y_true,z.p_dense,0.001)
d_best_t=float(d_best.threshold)
d_tuned=metrics(z.y_true,z.p_dense,d_best_t)
d_scan.to_csv(OUT/'densenet_oof_threshold_scan_001.csv',index=False)

# Coarse blend search.
coarse_rows=[]
for wd in np.round(np.arange(0.0,1.0001,0.05),2):
    p=wd*z.p_dense.to_numpy()+(1-wd)*z.p_xception.to_numpy()
    for t in np.round(np.arange(0.10,0.9001,0.005),3):
        mm=metrics(z.y_true,p,t)
        coarse_rows.append({'dense_weight':float(wd),'xception_weight':float(1-wd),**mm})
coarse=pd.DataFrame(coarse_rows)
coarse.to_csv(OUT/'ensemble_coarse_grid.csv',index=False)
cb=coarse.sort_values(['P','dense_weight','threshold'],ascending=[False,True,True]).iloc[0]
print('Coarse best:',cb.to_dict(), flush=True)

# Refine weight and threshold around the coarse optimum.
w0=float(cb.dense_weight); t0=float(cb.threshold)
weights=np.round(np.arange(max(0,w0-0.06),min(1,w0+0.06)+0.0001,0.005),3)
thresholds=np.round(np.arange(max(0.05,t0-0.03),min(0.95,t0+0.03)+0.0001,0.001),3)

fine_rows=[]
for wd in weights:
    p=wd*z.p_dense.to_numpy()+(1-wd)*z.p_xception.to_numpy()
    for t in thresholds:
        mm=metrics(z.y_true,p,t)
        fine_rows.append({'dense_weight':float(wd),'xception_weight':float(1-wd),**mm})
fine=pd.DataFrame(fine_rows)
fine.to_csv(OUT/'ensemble_fine_grid.csv',index=False)
fb=fine.sort_values(['P','dense_weight','threshold'],ascending=[False,True,True]).iloc[0]

best_w=float(fb.dense_weight)
best_t=float(fb.threshold)
blend=best_w*z.p_dense.to_numpy()+(1-best_w)*z.p_xception.to_numpy()
ens=metrics(z.y_true,blend,best_t)

# Fold-level stability of the selected global ensemble rule.
ens_folds=[]
for fold in range(5):
    mask=z.x_fold.astype(int).to_numpy()==fold
    ens_folds.append({'fold':fold,**metrics(z.y_true.to_numpy()[mask],blend[mask],best_t)})
ens_fold_df=pd.DataFrame(ens_folds)
ens_fold_df.to_csv(OUT/'ensemble_fold_metrics.csv',index=False)

ensemble_oof=pd.DataFrame({
    'filename':z.filename,
    'y_true':z.y_true,
    'fold':z.x_fold.astype(int),
    'p_dense':z.p_dense,
    'p_xception':z.p_xception,
    'p_ensemble':blend
})
ensemble_oof.to_csv(OUT/'ensemble_oof_predictions.csv',index=False)

print('DenseNet OOF fine-tuned:',d_tuned, flush=True)
print('Xception OOF fine-tuned:',x_tuned, flush=True)
print('BEST ENSEMBLE:',{'dense_weight':best_w,'xception_weight':1-best_w,'threshold':best_t,**ens}, flush=True)
display(ens_fold_df)


In [ ]:
print('>>> CELL 11 START: save final summary', flush=True)
summary={
    'xception_model':'xception41p.ra3_in1k',
    'pretrained':'ImageNet',
    'image_size':IMAGE_SIZE,
    'max_epochs':EPOCHS,
    'patience':PATIENCE,
    'fold_P_mean':float(fold_table.P.mean()),
    'fold_P_std':float(fold_table.P.std(ddof=1)),
    'xception_oof_at_0_5':x_base,
    'xception_oof_best_threshold':x_best_t,
    'xception_oof_tuned':x_tuned,
    'densenet_oof_best_threshold_fine':d_best_t,
    'densenet_oof_tuned_fine':d_tuned,
    'ensemble':{
        'dense_weight':best_w,
        'xception_weight':float(1-best_w),
        'threshold':best_t,
        'metrics':ens,
        'fold_P_mean':float(ens_fold_df.P.mean()),
        'fold_P_std':float(ens_fold_df.P.std(ddof=1))
    },
    'holdout_used':False
}
with open(OUT/'oof_ensemble_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2), flush=True)
print('\nSaved to:',OUT, flush=True)
print('SEALED HOLDOUT REMAINS UNUSED:',len(holdout), flush=True)


## Decision rule after this notebook

- Nếu Xception OOF ổn định và DenseNet+Xception ensemble tăng P trên toàn OOF: giữ cả hai vào finalist pool.
- Nếu gain chỉ xuất hiện ở một fold hoặc ensemble fold std tăng mạnh: chưa khóa ensemble; ưu tiên resolution/preprocessing ablation.
- Holdout 770 ảnh vẫn chưa được mở.
